# SaemPhysics: render a reviewed lesson draft in Google Colab

Codespaces/GitHub is the source of truth for code and reviewed metadata. This notebook clones that source, validates the approved first-batch plan, compiles an original kinematics example, and renders Manim visual drafts. MP4s and render reports are saved to Google Drive. This does **not** constitute final physics approval or permit publication of source-paper content.

In [ ]:
from pathlib import Path
import subprocess

REPOSITORY = "https://github.com/saemsir/physics.git"
REVISION = "main"  # Prefer a reviewed commit SHA for repeatable production renders.
ROOT = Path("/content/physics")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY, str(ROOT)], check=True)
else:
    print(f"Using existing checkout: {ROOT}")
subprocess.run(["git", "-C", str(ROOT), "fetch", "--depth", "1", "origin", REVISION], check=True)
subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", "FETCH_HEAD"], check=True)
%cd /content/physics
subprocess.run(["git", "rev-parse", "HEAD"], check=True)

## Install the renderer

This can take several minutes on a fresh Colab runtime. The Python requirements are pinned in the repository; system packages provide Cairo, Pango, LaTeX, and FFmpeg.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg libcairo2-dev libpango1.0-dev texlive-latex-extra texlive-fonts-recommended dvisvgm
!python -m pip install -q -r requirements.txt

## Select and validate the batch

Choose the reviewed batch JSON below. It produces exactly ten 60-second vertical briefs and ten 180-second horizontal briefs. Approval for storyboarding is not exam-source verification or final physics-release approval.

In [ ]:
BATCH_MANIFEST = "data/video_batches/batch_001.json"  # Change this to the reviewed batch JSON path.
!python -m content_engine batch-plan --manifest {BATCH_MANIFEST} --output /content/batch_plan.json
import json
batch_plan = json.loads(Path("/content/batch_plan.json").read_text())
batch_input = json.loads(Path(BATCH_MANIFEST).read_text())
BATCH_ID = batch_input["batch_id"]
PREVIEW_VERTICAL_ID = f"{batch_input['episodes'][0]['id']}-vertical"
PREVIEW_HORIZONTAL_ID = f"{batch_input['episodes'][0]['id']}-horizontal"
print(f"Plan status: {batch_plan['status']}; deliverables: {len(batch_plan['deliverables'])}")

## Build the linked database and compile a lesson draft

The demo is an original engine test case, not a question from the IIT-JEE archive. Replace it only with a question whose source, rights, wording, concept tags, assumptions, and answer have been reviewed.

In [ ]:
!python -m content_engine init
!python -m content_engine summary
!python -m content_engine compile-lesson --problem data/kinematics_demo.json

## Save batch exports to Google Drive

The 20 videos total 40 minutes of runtime. Render a preview first, then render vertical and horizontal versions in separate resumable passes. The renderer verifies frame size and duration and writes a SHA-256 report. Draft MP4s contain branded visuals and on-screen explanations; they do not yet contain recorded narration, final captions, or release approval.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
OUTPUT_ROOT = Path('/content/drive/MyDrive/SaemPhysics') / BATCH_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT_STR = str(OUTPUT_ROOT)

### Render two representative previews

Watch both aspect ratios before starting the full batch. Text branding is used because no image logo was supplied.

In [ ]:
!python tools/render_video_batch.py --manifest {BATCH_MANIFEST} --quality preview --video-ids {PREVIEW_VERTICAL_ID} {PREVIEW_HORIZONTAL_ID} --output-dir {OUTPUT_ROOT_STR}/preview
from IPython.display import Video, display
preview_dir = OUTPUT_ROOT / 'preview'
for video_id in (PREVIEW_VERTICAL_ID, PREVIEW_HORIZONTAL_ID):
    display(Video(str(preview_dir / f'{video_id}.mp4'), embed=True))

### Compare the first animation with its timed narration script

Cue timecodes match storyboard beat boundaries, not frame-level gestures. Read the script while replaying each silent preview to review pacing and visual alignment. Audio is not yet mixed into the MP4; record narration separately and keep the raw take private.

In [ ]:
import html
from IPython.display import HTML, Video, display
narration_path = Path('episodes/batch_001/narration.json')
narration = json.loads(narration_path.read_text())
if narration['batch_id'] != BATCH_ID:
    raise ValueError(f"Narration batch {narration['batch_id']} does not match selected batch {BATCH_ID}")
episode_script = next(item for item in narration['episodes'] if item['episode_id'] == batch_input['episodes'][0]['id'])
for format_name, video_id in (('vertical', PREVIEW_VERTICAL_ID), ('horizontal', PREVIEW_HORIZONTAL_ID)):
    print(f"{format_name.title()} narration cues — read along while the preview plays")
    rows = []
    for cue in episode_script[format_name]:
        start, end = narration['timecodes_seconds'][format_name][cue['beat']]
        rows.append(f"<tr><td>{start:02d}–{end:02d}s</td><td>{html.escape(cue['beat'])}</td><td>{html.escape(cue['text'])}</td></tr>")
    display(HTML('<table border="1" cellpadding="6"><tr><th>Time</th><th>Beat</th><th>Spoken line</th></tr>' + ''.join(rows) + '</table>'))
    display(Video(str(preview_dir / f'{video_id}.mp4'), embed=True))

### Render the production-sized draft batch

Run these separately. If Colab disconnects, rerun with `--resume`; completed MP4s are checked before reuse. Inspect every final video and add recorded narration/captions in post-production before release.

In [ ]:
!python tools/render_video_batch.py --manifest {BATCH_MANIFEST} --quality high --formats vertical --resume --output-dir {OUTPUT_ROOT_STR}/vertical

In [ ]:
!python tools/render_video_batch.py --manifest {BATCH_MANIFEST} --quality high --formats horizontal --resume --output-dir {OUTPUT_ROOT_STR}/horizontal

## Render and inspect

This is a low-quality preview for review, not a release export. Check the model assumptions, numerical result, axes, labels, timing, and legibility before approval.

In [ ]:
!python -m manim -ql episodes/kinematics_001/scenes.py KinematicsMotion --media_dir /content/physics-render
from IPython.display import Video, display
renders = list(Path("/content/physics-render").rglob("KinematicsMotion.mp4"))
if not renders:
    raise FileNotFoundError("Manim completed without producing KinematicsMotion.mp4")
display(Video(str(renders[0]), embed=True))

## Download a preview copy (optional)

Keep the reviewed source, lesson IR, and approval decisions in Git. Keep large generated renders outside normal Git history.

In [ ]:
from google.colab import files
files.download(str(renders[0]))